# Phase 4 — Replenishment Optimization

In this phase, we combine future demand forecasts with current inventory information to support replenishment decisions.

The goal is to estimate stock coverage, identify inventory risk, and recommend replenishment quantities based on expected demand.

## 4.1 Load Inventory and Forecast Summary

We start by loading:
- Current inventory information
- The 14-day forecast summary generated in Phase 3

These datasets will be combined using `product_id`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

inventory = pd.read_csv("../data/inventory.csv")

forecast_summary = pd.read_csv(
    "../data/forecast_summary_14d.csv"
)

inventory.head()

,product_id,current_stock,reorder_point,max_stock
0,P001,12,5,30
1,P002,8,10,40
2,P003,15,8,35
3,P004,4,5,20
4,P005,20,6,30


In [ ]:
print("Inventory shape:", inventory.shape)
print("Forecast summary shape:", forecast_summary.shape)

print("\nInventory columns:")
print(inventory.columns.tolist())

print("\nForecast columns:")
print(forecast_summary.columns.tolist())

print("\nMissing values in inventory:")
print(inventory.isna().sum())

print("\nMissing values in forecast summary:")
print(forecast_summary.isna().sum())

Inventory shape: (10, 4)
Forecast summary shape: (10, 4)

Inventory columns:
['product_id', 'current_stock', 'reorder_point', 'max_stock']

Forecast columns:
['product_id', 'total_forecast_14d', 'avg_daily_forecast', 'max_daily_forecast']

Missing values in inventory:
product_id       0
current_stock    0
reorder_point    0
max_stock        0
dtype: int64

Missing values in forecast summary:
product_id            0
total_forecast_14d    0
avg_daily_forecast    0
max_daily_forecast    0
dtype: int64


## 4.2 Combine Inventory with Forecast Data

To make replenishment decisions, inventory information and forecast information must be available in the same table.

We merge the two datasets using `product_id`.

This creates one row per product containing both:
- Current stock information
- Expected future demand

In [ ]:
replenishment = inventory.merge(
    forecast_summary,
    on="product_id",
    how="left"
)

replenishment

,product_id,current_stock,reorder_point,max_stock,total_forecast_14d,avg_daily_forecast,max_daily_forecast
0,P001,12,5,30,90.83,6.49,7.53
1,P002,8,10,40,265.89,18.99,22.79
2,P003,15,8,35,157.90,11.28,12.76
3,P004,4,5,20,60.62,4.33,4.76
4,P005,20,6,30,128.99,9.21,10.42
5,P006,3,5,15,99.84,7.13,8.37
6,P007,7,4,12,37.59,2.68,3.42
7,P008,25,8,40,199.57,14.26,16.23
8,P009,18,5,25,80.96,5.78,6.57
9,P010,6,8,30,164.30,11.74,14.57


## 4.3 Calculate Forecasted Days of Supply

Days of Supply estimates how many days the current inventory can support based on forecasted average daily demand.

Formula:

Days of Supply = Current Stock / Average Daily Forecast

This metric helps identify products that may run out of stock soon if expected demand continues at the forecasted level.

In [ ]:
replenishment["days_of_supply"] = np.where(
    replenishment["avg_daily_forecast"] > 0,
    replenishment["current_stock"] / replenishment["avg_daily_forecast"],
    np.nan
)

replenishment["days_of_supply"] = (
    replenishment["days_of_supply"]
    .round(2)
)

replenishment[
    [
        "product_id",
        "current_stock",
        "avg_daily_forecast",
        "days_of_supply"
    ]
]

,product_id,current_stock,avg_daily_forecast,days_of_supply
0,P001,12,6.49,1.85
1,P002,8,18.99,0.42
2,P003,15,11.28,1.33
3,P004,4,4.33,0.92
4,P005,20,9.21,2.17
5,P006,3,7.13,0.42
6,P007,7,2.68,2.61
7,P008,25,14.26,1.75
8,P009,18,5.78,3.11
9,P010,6,11.74,0.51


## 4.4 Classify Inventory Risk

To make stock coverage easier to interpret, we classify each product into a simple inventory risk category based on forecasted Days of Supply.

Baseline rules:
- High Risk: less than 2 days of supply
- Medium Risk: from 2 to less than 5 days
- Low Risk: 5 days or more

These thresholds are simple educational baseline rules and can later be improved using supplier lead time, service level, and business constraints.

In [ ]:
def classify_inventory_risk(days):
    if pd.isna(days):
        return "No Forecast"
    elif days < 2:
        return "High Risk"
    elif days < 5:
        return "Medium Risk"
    else:
        return "Low Risk"


replenishment["inventory_risk"] = (
    replenishment["days_of_supply"]
    .apply(classify_inventory_risk)
)

replenishment[
    [
        "product_id",
        "current_stock",
        "avg_daily_forecast",
        "days_of_supply",
        "inventory_risk"
    ]
]

,product_id,current_stock,avg_daily_forecast,days_of_supply,inventory_risk
0,P001,12,6.49,1.85,High Risk
1,P002,8,18.99,0.42,High Risk
2,P003,15,11.28,1.33,High Risk
3,P004,4,4.33,0.92,High Risk
4,P005,20,9.21,2.17,Medium Risk
5,P006,3,7.13,0.42,High Risk
6,P007,7,2.68,2.61,Medium Risk
7,P008,25,14.26,1.75,High Risk
8,P009,18,5.78,3.11,Medium Risk
9,P010,6,11.74,0.51,High Risk


## 4.5 Forecast-Based Reorder Decision

The original replenishment baseline used only the current stock level and reorder point.

In this phase, we improve the decision by also considering forecasted Days of Supply.

A product is marked for replenishment when:
- Current stock is at or below the reorder point, OR
- Forecasted Days of Supply is less than 5 days

This combines current inventory status with expected future demand.

In [ ]:
replenishment["needs_reorder"] = (
    (replenishment["current_stock"] <= replenishment["reorder_point"])
    |
    (replenishment["days_of_supply"] < 5)
)

replenishment[
    [
        "product_id",
        "current_stock",
        "reorder_point",
        "days_of_supply",
        "inventory_risk",
        "needs_reorder"
    ]
]

,product_id,current_stock,reorder_point,days_of_supply,inventory_risk,needs_reorder
0,P001,12,5,1.85,High Risk,True
1,P002,8,10,0.42,High Risk,True
2,P003,15,8,1.33,High Risk,True
3,P004,4,5,0.92,High Risk,True
4,P005,20,6,2.17,Medium Risk,True
5,P006,3,5,0.42,High Risk,True
6,P007,7,4,2.61,Medium Risk,True
7,P008,25,8,1.75,High Risk,True
8,P009,18,5,3.11,Medium Risk,True
9,P010,6,8,0.51,High Risk,True


## 4.6 Calculate Recommended Reorder Quantity

After identifying products that need replenishment, we estimate how many units should be ordered.

The recommendation considers:
- Current stock
- Forecasted demand over the next 14 days
- Maximum stock capacity

The reorder quantity is limited by `max_stock` to avoid recommending more inventory than the configured stock target.

In [ ]:
replenishment["forecast_based_reorder_qty"] = np.where(
    replenishment["needs_reorder"],
    np.minimum(
        np.ceil(
            replenishment["total_forecast_14d"]
            - replenishment["current_stock"]
        ),
        replenishment["max_stock"] - replenishment["current_stock"]
    ),
    0
)

replenishment["forecast_based_reorder_qty"] = (
    replenishment["forecast_based_reorder_qty"]
    .clip(lower=0)
    .astype(int)
)

replenishment[
    [
        "product_id",
        "current_stock",
        "total_forecast_14d",
        "max_stock",
        "needs_reorder",
        "forecast_based_reorder_qty"
    ]
]

,product_id,current_stock,total_forecast_14d,max_stock,needs_reorder,forecast_based_reorder_qty
0,P001,12,90.83,30,True,18
1,P002,8,265.89,40,True,32
2,P003,15,157.90,35,True,20
3,P004,4,60.62,20,True,16
4,P005,20,128.99,30,True,10
5,P006,3,99.84,15,True,12
6,P007,7,37.59,12,True,5
7,P008,25,199.57,40,True,15
8,P009,18,80.96,25,True,7
9,P010,6,164.30,30,True,24


## 4.7 Assign Replenishment Priority

Since multiple products may require replenishment at the same time, we assign a priority level based on forecasted Days of Supply.

Priority rules:
- Critical: less than 1 day of supply
- High: from 1 to less than 2 days
- Medium: from 2 to less than 5 days
- Low: 5 days or more

This helps identify which products should be replenished first.

In [ ]:
def assign_replenishment_priority(days):
    if pd.isna(days):
        return "Unknown"
    elif days < 1:
        return "Critical"
    elif days < 2:
        return "High"
    elif days < 5:
        return "Medium"
    else:
        return "Low"


replenishment["replenishment_priority"] = (
    replenishment["days_of_supply"]
    .apply(assign_replenishment_priority)
)

replenishment[
    [
        "product_id",
        "days_of_supply",
        "inventory_risk",
        "needs_reorder",
        "forecast_based_reorder_qty",
        "replenishment_priority"
    ]
]

,product_id,days_of_supply,inventory_risk,needs_reorder,forecast_based_reorder_qty,replenishment_priority
0,P001,1.85,High Risk,True,18,High
1,P002,0.42,High Risk,True,32,Critical
2,P003,1.33,High Risk,True,20,High
3,P004,0.92,High Risk,True,16,Critical
4,P005,2.17,Medium Risk,True,10,Medium
5,P006,0.42,High Risk,True,12,Critical
6,P007,2.61,Medium Risk,True,5,Medium
7,P008,1.75,High Risk,True,15,High
8,P009,3.11,Medium Risk,True,7,Medium
9,P010,0.51,High Risk,True,24,Critical


## 4.8 Build the Final Replenishment Decision Table

To make the replenishment results easier to use, we create a compact decision table containing the most important inventory and forecasting indicators.

The table is sorted by replenishment priority so that the most urgent products appear first.

Key decision fields include:
- Current stock
- Forecasted Days of Supply
- Inventory risk
- Reorder decision
- Recommended reorder quantity
- Replenishment priority

In [ ]:
priority_order = {
    "Critical": 0,
    "High": 1,
    "Medium": 2,
    "Low": 3,
    "Unknown": 4
}

replenishment["priority_rank"] = (
    replenishment["replenishment_priority"]
    .map(priority_order)
)

decision_table = (
    replenishment[
        [
            "product_id",
            "current_stock",
            "reorder_point",
            "max_stock",
            "avg_daily_forecast",
            "days_of_supply",
            "inventory_risk",
            "needs_reorder",
            "forecast_based_reorder_qty",
            "replenishment_priority",
            "priority_rank"
        ]
    ]
    .sort_values(
        ["priority_rank", "days_of_supply"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

decision_table

,product_id,current_stock,reorder_point,max_stock,avg_daily_forecast,days_of_supply,inventory_risk,needs_reorder,forecast_based_reorder_qty,replenishment_priority,priority_rank
0,P002,8,10,40,18.99,0.42,High Risk,True,32,Critical,0
1,P006,3,5,15,7.13,0.42,High Risk,True,12,Critical,0
2,P010,6,8,30,11.74,0.51,High Risk,True,24,Critical,0
3,P004,4,5,20,4.33,0.92,High Risk,True,16,Critical,0
4,P003,15,8,35,11.28,1.33,High Risk,True,20,High,1
5,P008,25,8,40,14.26,1.75,High Risk,True,15,High,1
6,P001,12,5,30,6.49,1.85,High Risk,True,18,High,1
7,P005,20,6,30,9.21,2.17,Medium Risk,True,10,Medium,2
8,P007,7,4,12,2.68,2.61,Medium Risk,True,5,Medium,2
9,P009,18,5,25,5.78,3.11,Medium Risk,True,7,Medium,2


## 4.9 Save Replenishment Results

The final replenishment decision table is saved for use in later phases.

This output contains the most important inventory and forecasting indicators, including:
- Current stock
- Forecasted Days of Supply
- Inventory risk
- Reorder decision
- Recommended reorder quantity
- Replenishment priority

The saved file will be used as an input for simulation and decision-support visualization.

In [ ]:
decision_table.to_csv(
    "../data/replenishment_decisions.csv",
    index=False
)

print("Replenishment decisions saved successfully.")

Replenishment decisions saved successfully.


In [ ]:
final_decision_table = decision_table.drop(
    columns=["priority_rank"]
)

final_decision_table.to_csv(
    "../data/replenishment_decisions.csv",
    index=False
)

final_decision_table

,product_id,current_stock,reorder_point,max_stock,avg_daily_forecast,days_of_supply,inventory_risk,needs_reorder,forecast_based_reorder_qty,replenishment_priority
0,P002,8,10,40,18.99,0.42,High Risk,True,32,Critical
1,P006,3,5,15,7.13,0.42,High Risk,True,12,Critical
2,P010,6,8,30,11.74,0.51,High Risk,True,24,Critical
3,P004,4,5,20,4.33,0.92,High Risk,True,16,Critical
4,P003,15,8,35,11.28,1.33,High Risk,True,20,High
5,P008,25,8,40,14.26,1.75,High Risk,True,15,High
6,P001,12,5,30,6.49,1.85,High Risk,True,18,High
7,P005,20,6,30,9.21,2.17,Medium Risk,True,10,Medium
8,P007,7,4,12,2.68,2.61,Medium Risk,True,5,Medium
9,P009,18,5,25,5.78,3.11,Medium Risk,True,7,Medium


## Phase 4 Conclusion

Phase 4 transformed demand forecasts into practical inventory replenishment decisions.

By combining current inventory data with the 14-day demand forecast, we calculated forecasted Days of Supply, classified inventory risk, identified products that require replenishment, estimated reorder quantities, and assigned replenishment priorities.

The final decision table provides a clear product-level view of:
- Current stock
- Forecasted stock coverage
- Inventory risk
- Reorder requirement
- Recommended reorder quantity
- Replenishment priority

The current replenishment logic is a transparent forecast-informed baseline rather than a full optimization model.

The next phase evaluates these replenishment decisions through inventory simulation and Digital Twin scenarios.